# What we recommend

The short list. Each recommendation is rule-triggered from the data — no judgement call, no editorial filter. Severity follows the Bridge-OD evidence report ([reports/07_bridge_od.md](../../reports/07_bridge_od.md)):

- **HIGH** — the data clears several thresholds at once, action is well-justified.
- **MEDIUM** — one or two thresholds; action should follow a local conversation.
- **INFO** — context for the council, not an immediate ask.

In [1]:
from pathlib import Path
import os, sys, warnings

import pandas as pd

REPO = Path.cwd()
while not (REPO / 'leonia_traffic').is_dir() and REPO.parent != REPO:
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
warnings.filterwarnings('ignore')

from leonia_traffic.analysis import congestion as cg, od_cutthrough as oc
from leonia_traffic.analysis.equity import equity_exposure_index
from leonia_traffic.analysis.jurisdiction import (
    annotate_in_leonia, filter_segments_to_leonia,
)
from leonia_traffic.analysis.recommendations import (
    generate_recommendations,
)
from leonia_traffic.data.bridge_od_loader import (
    load_bridge_attributes, load_bridge_od,
)
from leonia_traffic.data.congestion_loader import (
    load_congestion, load_congestion_zones, summarize_link_reliability,
)

od_df = load_bridge_od(); attr_df = load_bridge_attributes()
cdf = load_congestion(); czones = load_congestion_zones()
peak_imbalance = oc.gateway_peak_imbalance(od_df)
circuity = oc.cutthrough_index_from_circuity(attr_df)
exposure = equity_exposure_index(attr_df)
summary = annotate_in_leonia(summarize_link_reliability(cdf), czones)
delay = annotate_in_leonia(cg.delay_hotspot_ranking(cdf), czones)
summary_l = filter_segments_to_leonia(summary, czones)
delay_l = filter_segments_to_leonia(delay, czones)
per_street_path = Path('data/processed/leonia_streets_cutthrough_index.parquet')
per_street_df = (
    pd.read_parquet(per_street_path) if per_street_path.exists() else None
)

recs = generate_recommendations(
    peak_imbalance_df=peak_imbalance, circuity_df=circuity,
    delay_df=delay_l, summary_df=summary_l, exposure_df=exposure,
    per_street_df=per_street_df,
)

## Summary

In [2]:
from IPython.display import Markdown
by_sev = pd.Series([r.severity for r in recs]).value_counts()
lines = [f'- **{sev.upper()}**: {n}' for sev, n in by_sev.items()]
Markdown(
    f'**{len(recs)}** rule-triggered recommendations were generated from the '
    f'current data lake:\n\n' + '\n'.join(lines)
)

**27** rule-triggered recommendations were generated from the current data lake:

- **MEDIUM**: 17
- **HIGH**: 8
- **INFO**: 2

## High-severity items

In [3]:
highs = [r for r in recs if r.severity == 'high']
if not highs:
    md = '_No high-severity items triggered in the current snapshot._\n'
else:
    parts = []
    for r in highs:
        bits = ', '.join(
            f'**{k}**: {v}' for k, v in r.metrics.items() if v is not None
        )
        parts.append(
            f'### {r.rank}. {r.target}  \n'
            f'_Rule_: `{r.rule}`  \n'
            f'{r.rationale}  \n'
            f'{bits}\n'
        )
    md = '\n'.join(parts)
Markdown(md)

### 1. Fort Lee Road  
_Rule_: `primary_mitigation_candidate`  
Peak-AM weekday volume of 694 trips/day and weekday/weekend ratio of 10.1× indicate concentrated commuter cut-through; this is the primary candidate for traffic-calming or routing changes.  
**peak_am_weekday**: 693, **weekend_peak_am**: 68, **ratio**: 10.1

### 2. Willow Tree Road  
_Rule_: `residential_cutthrough_candidate`  
Composite cut-through index 0.59 (1,257 weekday Visitor trips/day, 61% with home ≥3 mi away). Residential street with strong direct evidence of regional pass-through traffic; primary calming candidate.  
**osm_way**: 3356462, **index**: 0.59, **wd_vol**: 1257, **non_local**: 0.61, **wd_to_sat**: 3.05

### 3. Broad Avenue  
_Rule_: `residential_cutthrough_candidate`  
Composite cut-through index 0.56 (12,762 weekday Visitor trips/day, 65% with home ≥3 mi away). Residential street with strong direct evidence of regional pass-through traffic; primary calming candidate.  
**osm_way**: 10030557, **index**: 0.56, **wd_vol**: 12762, **non_local**: 0.65, **wd_to_sat**: 1.26

### 4. Schor Avenue  
_Rule_: `residential_cutthrough_candidate`  
Composite cut-through index 0.55 (740 weekday Visitor trips/day, 52% with home ≥3 mi away). Residential street with strong direct evidence of regional pass-through traffic; primary calming candidate.  
**osm_way**: 17834065, **index**: 0.55, **wd_vol**: 740, **non_local**: 0.52, **wd_to_sat**: 2.88

### 5. Pine Hill Road  
_Rule_: `residential_cutthrough_candidate`  
Composite cut-through index 0.51 (780 weekday Visitor trips/day, 63% with home ≥3 mi away). Residential street with strong direct evidence of regional pass-through traffic; primary calming candidate.  
**osm_way**: 532511, **index**: 0.51, **wd_vol**: 780, **non_local**: 0.63, **wd_to_sat**: 1.41

### 6. Main Street  
_Rule_: `residential_cutthrough_candidate`  
Composite cut-through index 0.51 (34,644 weekday Visitor trips/day, 63% with home ≥3 mi away). Residential street with strong direct evidence of regional pass-through traffic; primary calming candidate.  
**osm_way**: 11099916, **index**: 0.51, **wd_vol**: 34644, **non_local**: 0.63, **wd_to_sat**: 1.07

### 7. Nordhoff Drive  
_Rule_: `residential_cutthrough_candidate`  
Composite cut-through index 0.48 (919 weekday Visitor trips/day, 58% with home ≥3 mi away). Residential street with strong direct evidence of regional pass-through traffic; primary calming candidate.  
**osm_way**: 8998330, **index**: 0.48, **wd_vol**: 919, **non_local**: 0.58, **wd_to_sat**: 1.45

### 8. Fort Lee Road  
_Rule_: `failing_corridor_exclude_from_diversion`  
Worst-hour TTI 2.01 and Buffer Index 8.81 show this corridor is already failing; any mitigation that diverts traffic onto it would worsen conditions.  
**tti**: 2.01, **buffer**: 8.81, **vhd**: 71.7


## Medium-severity items

In [4]:
meds = [r for r in recs if r.severity == 'medium']
if not meds:
    md = '_No medium-severity items triggered._\n'
else:
    rows = []
    for r in meds:
        rows.append({'#': r.rank, 'Target': r.target,
                     'Rule': r.rule, 'Why': r.rationale})
    Markdown('See table below.')
    md = '_See table below._'
Markdown(md)
if meds:
    pd.DataFrame(rows)
else:
    pd.DataFrame()

In [5]:
if meds:
    display_df = pd.DataFrame([
        {'#': r.rank, 'Target': r.target, 'Rule': r.rule, 'Why': r.rationale}
        for r in meds
    ])
    display_df
else:
    pd.DataFrame()

## Info-only callouts

In [6]:
infos = [r for r in recs if r.severity == 'info']
if not infos:
    Markdown('_No info callouts._')
else:
    pd.DataFrame([
        {'#': r.rank, 'Target': r.target, 'Rule': r.rule, 'Note': r.rationale}
        for r in infos
    ])

## How these were generated

Each recommendation comes from a named, documented rule in [leonia_traffic/analysis/recommendations.py](../../leonia_traffic/analysis/recommendations.py). The rule code is small and auditable — anyone can read it and decide whether the threshold makes sense. We deliberately do not use a black-box model.

The diversion-impact dev notebook ([notebooks/dev/03_recommendations.ipynb](../dev/03_recommendations.ipynb)) re-runs each high/medium item through the traffic-assignment engine and flags any > 20 % spillover. Use it before committing to any single intervention.

## Limitations

- The Bridge-OD product captures the **morning commute toward the George Washington Bridge** — the dominant single cut-through generator but not the only one. Recommendations grounded in school-drop-off or evening flows would need additional data.
- All recommendations are filtered to streets under **Borough of Leonia jurisdiction**. State and federal facilities (NJ Turnpike, Route 46, GWB approaches) are excluded; addressing them requires NJDOT / Port Authority coordination.